In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# ── Sample dataset ─────────────────────────────────────────
data = {
    "student_id": ["S001","S002","S003","S004","S005",
                   "S006","S007","S008","S009","S010",
                   "S011","S012","S013","S014","S015",
                   "S016","S017","S018","S019","S020"],
    "arm":        ["control"]*10 + ["treatment"]*10,
    "grade":      [8,9,8,10,9,8,10,9,8,10,
                   9,8,10,8,9,10,8,9,10,8],
    "pre_score":  [62,58,71,65,54,68,60,57,73,61,
                   63,56,70,64,55,67,59,58,72,62],
    "post_score": [60,55,69,63,52,65,58,56,71,59,
                   68,62,77,70,61,74,65,65,79,69],
    "dropped_out":[1,0,0,1,1,0,1,0,0,1,
                   0,0,0,0,1,0,0,0,0,0],
    "geo_index":  [0.22,0.31,0.18,0.28,0.35,
                   0.24,0.29,0.20,0.33,0.27,
                   0.21,0.30,0.19,0.26,0.34,
                   0.23,0.28,0.22,0.32,0.25]
}
df = pd.DataFrame(data)
df["delta_score"] = df["post_score"] - df["pre_score"]

ctrl = df[df["arm"] == "control"]
trt  = df[df["arm"] == "treatment"]

# ── Test 1: Paired t-test (within each arm) ────────────────
# H0: mean delta = 0 (no change from baseline)
t_ctrl, p_ctrl = stats.ttest_rel(ctrl["post_score"], ctrl["pre_score"])
t_trt,  p_trt  = stats.ttest_rel(trt["post_score"],  trt["pre_score"])

print(f"Control  — t={t_ctrl:.3f}, p={p_ctrl:.4f}")
print(f"Treatment— t={t_trt:.3f},  p={p_trt:.4f}")

# ── Test 2: Independent t-test (delta: treatment vs control)
# H0: mean(delta_trt) == mean(delta_ctrl)
t_ind, p_ind = stats.ttest_ind(
    trt["delta_score"],
    ctrl["delta_score"],
    equal_var=False   # Welch's — safer default
)

# Cohen's d
pool_std = np.sqrt((trt["delta_score"].std()**2 +
                     ctrl["delta_score"].std()**2) / 2)
cohens_d = (trt["delta_score"].mean() -
            ctrl["delta_score"].mean()) / pool_std

print(f"Independent — t={t_ind:.3f}, p={p_ind:.4f}, d={cohens_d:.3f}")